# Texas Childcare: Compliance Records vs. Family Experience
## Notebook 04 — Cross-Model Comparison and Findings

**Author:** Rakesh Chandrasekaran
**Program:** Professional Certificate in ML and AI, UC Berkeley
**Date:** June 2026

---

## Research Question

**Do Texas HHSC compliance records and families' Google reviews agree on which daycare centers are high quality — and can a parent rely on compliance records alone to make an informed childcare decision?**

---

## Cities in Scope
Austin · Houston · San Antonio · Dallas · Fort Worth

---

### This Notebook
Consolidates results from the four-notebook pipeline (Notebooks 00-03) into a single cross-model comparison. Presents the final answer to the research question across all three stakeholders — parents, compliance officers, and daycare operators.

### Notebook Structure
0. Setup and Data Loading
1. Research Journey — What Was Done in Each Notebook
2. Final Model Selection — Which Models Are Compared and Why
3. Cross-Model Comparison
    - 3.1 Summary Comparison Table
    - 3.2 ROC Curve Overlay (all 3 models)
    - 3.3 Predicted vs Actual Scatter Comparison
    - 3.4 City-Level Performance Analysis
4. What Each Framework Measures
    - 4.1 Compliance Framework vs Text Framework
    - 4.2 Word Coefficients — What Drives Ratings
5. Findings — Research Question Answer
6. Limitations
7. Future Work

## 0. Setup and Data Loading

In [ ]:
# ── Libraries ────────────────────────────────────────────────────────────
import os, pickle
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ── Visualisation ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="muted")

# ── Metrics ───────────────────────────────────────────────────────────────
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, confusion_matrix

# ── Directory configuration ───────────────────────────────────────────────
BASE_DIR  = os.getcwd()
IMAGE_DIR = os.path.join(BASE_DIR, "images", "comparison")
os.makedirs(IMAGE_DIR, exist_ok=True)
PROC_DIR  = os.path.join(BASE_DIR, "data", "processed")

# ── Load all model outputs (pickle) ───────────────────────────────────────
with open(os.path.join(PROC_DIR, "model1_outputs.pkl"),  "rb") as f: m1  = pickle.load(f)
with open(os.path.join(PROC_DIR, "model2a_outputs.pkl"), "rb") as f: m2a = pickle.load(f)
with open(os.path.join(PROC_DIR, "model2b_outputs.pkl"), "rb") as f: m2b = pickle.load(f)

# ── Unpack Model 1 ────────────────────────────────────────────────────────
rmse_ridge = m1["rmse_ridge"]; r2_ridge = m1["r2_ridge"]
rmse_xgb   = m1["rmse_xgb"];   r2_xgb   = m1["r2_xgb"];  m1_auc = m1["m1_auc"]
y1_pred_prob = m1["y1_pred_prob"]; y1_clf_test = m1["y1_clf_test"]

# ── Unpack Model 2A ───────────────────────────────────────────────────────
rmse_m2a = m2a["rmse_m2a"]; r2_m2a = m2a["r2_m2a"]; m2a_clf_auc = m2a["m2a_clf_auc"]
y2a_clf_pred_prob = m2a["y2a_clf_pred_prob"]; y2a_clf_test = m2a["y2a_clf_test"]
y2a_test_op_ids   = m2a["y2a_test_op_ids"]

# ── Unpack Model 2B ───────────────────────────────────────────────────────
rmse_2b_simple   = m2b["rmse_2b_simple"];   rmse_2b_recency = m2b["rmse_2b_recency"]
r2_2b_simple     = m2b["r2_2b_simple"]
auc_2b_simple    = m2b["auc_2b_simple"];    auc_2b_recency  = m2b["auc_2b_recency"]
center_labels    = m2b["center_labels"]
center_clf_simple = m2b["center_clf_simple"]
common_ids       = m2b["common_ids"]

print(f"All outputs loaded.")
print(f"  Model 1 : RMSE={rmse_xgb:.4f}  R²={r2_xgb:.4f}  AUC={m1_auc:.4f}")
print(f"  Model 2A: RMSE={rmse_m2a:.4f}  R²={r2_m2a:.4f}  AUC={m2a_clf_auc:.4f}")
print(f"  Model 2B: RMSE={rmse_2b_simple:.4f}  R²={r2_2b_simple:.4f}  AUC={auc_2b_simple:.4f}")

In [ ]:
# Pre-flight check — confirm all pkl outputs loaded correctly
required = {
    "m1":  ["rmse_xgb", "r2_xgb", "m1_auc", "y1_pred_prob", "y1_clf_test"],
    "m2a": ["rmse_m2a", "r2_m2a", "m2a_clf_auc", "y2a_clf_pred_prob", "y2a_clf_test", "y2a_test_op_ids"],
    "m2b": ["rmse_2b_simple", "rmse_2b_recency", "r2_2b_simple", "r2_2b_recency",
            "auc_2b_simple", "auc_2b_recency",
            "center_labels", "center_clf_simple", "common_ids"],
}
all_ok = True
for pkl_name, keys in required.items():
    pkg = eval(pkl_name)
    for key in keys:
        status = "OK" if key in pkg else "MISSING"
        if status == "MISSING": all_ok = False
        print(f"  {pkl_name}['{key}']  {status}")
print()
print("All inputs ready ✓" if all_ok else "MISSING INPUTS — check pkl files")

## 1. Research Journey — What Was Done in Each Notebook

This notebook is the final stage of a four-notebook pipeline. Each notebook built on the previous one to answer the research question progressively.

**[00-data-collection-preparation-eda.ipynb](00-data-collection-preparation-eda.ipynb)**

- Collected HHSC compliance records for 2,192 licensed daycare centers across Austin, Dallas, Fort Worth, Houston, and San Antonio
- Matched each center to its Google Places listing to retrieve star ratings and up to 5 review texts
- Key finding: compliance features show weak negative correlations with Google ratings (max correlation: 0.20)
- City effects explain more variance than any individual compliance feature
- A Ridge regression baseline explained only 6% of rating variance, establishing the compliance signal as weak

**[01-model1-compliance-xgboost.ipynb](01-model1-compliance-xgboost.ipynb)**

- Trained XGBoost regression and classification on 15 HHSC compliance features
- SHAP analysis revealed the top predictors are operational proxies (YEARS_IN_OPERATION, TOTAL_INSPECTIONS, ACCEPTS_SUBSIDIES), not direct quality measures
- Regression: RMSE=0.5555, R²=0.0579
- Classification: AUC=0.6441
- Compliance features correctly identified 59% of poor-quality centers

**[02-model2a-center-text.ipynb](02-model2a-center-text.ipynb)**

- Trained TF-IDF + Ridge (regression) and TF-IDF + Logistic Regression (classification) on center-level aggregated review text
- Used the same 1,327 centers and same train/test split as Model 1 for a direct comparison
- Linear models outperformed XGBoost on TF-IDF features for both tasks
- Regression: RMSE=0.4477, R²=0.3901
- Classification: AUC=0.8418
- Review text explained 39% of rating variance — 6x more than compliance data

**[03-model2b-individual-reviews.ipynb](03-model2b-individual-reviews.ipynb)**

- Tested whether the finding holds when training on individual reviews with center-aligned split and google_rating labels
- Simple mean aggregation outperformed recency weighted across both tasks
- Regression: RMSE=0.4933, R²=0.2597
- Classification: AUC=0.8034
- Confirms the text-over-compliance finding using an independent modeling approach

## 2. Final Model Selection

**Models included in the primary comparison:**

| Model | Approach | Notebook | Regression | Classification |
|---|---|---|---|---|
| Model 1 | Compliance features (XGBoost) | [Notebook 01](01-model1-compliance-xgboost.ipynb) | RMSE=0.5555, R²=0.0579 | AUC=0.6441 |
| Model 2A | Center-level text (TF-IDF + Ridge / LR) | [Notebook 02](02-model2a-center-text.ipynb) | RMSE=0.4477, R²=0.3901 | AUC=0.8418 |
| Model 2B.2 | Individual reviews, center-aligned (TF-IDF + Ridge / LR) | [Notebook 03](03-model2b-individual-reviews.ipynb) | RMSE=0.4933, R²=0.2597 | AUC=0.8034 |

**Models excluded from primary comparison and why:**

Model 2B.1 (individual review approach with `review_rating >= 4` labels) cannot be compared directly to Model 1 — it uses different labels, a review level test split, and predicts individual reviewer sentiment rather than center level quality. Its AUC=0.9915 reflects an easier task, not a superior model for the research question.

XGBoost variants were tested in both Notebook 02 and Notebook 03 for both regression and classification. Linear models (Ridge, Logistic Regression) outperformed XGBoost on high-dimensional sparse TF-IDF features in all four comparisons. XGBoost variants are excluded from the primary comparison.

For Model 2B.2, both simple mean and recency weighted aggregation were tested. Simple mean outperformed recency weighted on both RMSE (0.4933 vs 0.4964) and AUC (0.8034 vs 0.7638). With only 5 reviews per center, temporal weighting amplifies noise rather than improving signal. Simple mean aggregation is used for the primary comparison.

## 3. Cross-Model Comparison

### 3.1 Summary Comparison Table

In [ ]:
# Full comparison table across all three models
comparison = pd.DataFrame({
    "Model": [
        "Model 1 (Compliance Features)",
        "Model 2A (Center-Level Text)",
        "Model 2B.2 (Individual Reviews, Center-Aligned)"
    ],
    "RMSE": [rmse_xgb, rmse_m2a, rmse_2b_simple],
    "R²":   [r2_xgb,   r2_m2a,   r2_2b_simple],
    "AUC":  [m1_auc,   m2a_clf_auc, auc_2b_simple],
})
comparison["RMSE"] = comparison["RMSE"].map("{:.4f}".format)
comparison["R²"]   = comparison["R²"].apply(lambda x: "—" if x is None or (hasattr(x, "__float__") and __import__("math").isnan(float(x))) else f"{x:.4f}")
comparison["AUC"]  = comparison["AUC"].map("{:.4f}".format)
display(comparison.set_index("Model"))

In [ ]:
# Two-panel comparison bar chart: RMSE and AUC across all three models
models     = ['Model 1\n(Compliance)', 'Model 2A\n(Center Text)', 'Model 2B.2\n(Indiv. Reviews)']
rmse_vals  = [rmse_xgb, rmse_m2a, rmse_2b_simple]
auc_vals   = [m1_auc, m2a_clf_auc, auc_2b_simple]
colors     = ['coral', 'seagreen', 'steelblue']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# RMSE panel
bars = axes[0].bar(models, rmse_vals, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Regression — RMSE Comparison\n(lower is better)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('RMSE (star rating points)', fontsize=10)
axes[0].bar_label(bars, fmt='%.4f', padding=3, fontsize=10)
axes[0].set_ylim(0, max(rmse_vals) * 1.2)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# AUC panel
bars2 = axes[1].bar(models, auc_vals, color=colors, edgecolor='white', width=0.5)
axes[1].set_title('Classification — AUC Comparison\n(higher is better)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('AUC-ROC', fontsize=10)
axes[1].bar_label(bars2, fmt='%.4f', padding=3, fontsize=10)
axes[1].set_ylim(0, 1.1)
axes[1].axhline(y=0.5, color='grey', linestyle='--', linewidth=0.8, label='Random (AUC=0.5)')
axes[1].legend(fontsize=9)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle('Compliance Features vs Review Text — Final Model Comparison',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(IMAGE_DIR, 'comparison_all_3_models.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {os.path.join(IMAGE_DIR, "comparison_all_3_models.png")}')

**Model Comparison — Inference**

Review text substantially outperforms compliance records on both tasks.

For regression, Model 2A predicts center star ratings with 19.4% lower error than Model 1 (RMSE: 0.4477 vs 0.5555). Review text explains 39% of rating variance (R²=0.3901) compared to 6% for compliance records (R²=0.0579).

For classification, Model 2A correctly ranks excellent centers above poor ones 84% of the time (AUC=0.8418) compared to 64% for compliance records (AUC=0.6441).

Model 2B.2 confirms the finding using a different modeling approach — individual reviews with center aligned split produce RMSE=0.4933 and AUC=0.8034, both well above compliance performance. The consistency across two independent text models confirms the result is structural, not specific to one modeling choice.

**Why AUC rather than accuracy.** With 59.9% of centers rated HIGH_RATED, a model that predicts everything as HIGH_RATED achieves 59.9% accuracy while identifying zero poor-quality centers. AUC measures ranking quality regardless of class imbalance, making it the appropriate metric for this dataset. This choice is documented in Design Decision 2 in [01-model1-compliance-xgboost.ipynb](01-model1-compliance-xgboost.ipynb).

### 3.2 ROC Curve Overlay — All Three Models

In [ ]:
# ROC curve overlay: Model 1, Model 2A, Model 2B.2
fig, ax = plt.subplots(figsize=(9, 7))

fpr1, tpr1, _ = roc_curve(y1_clf_test, y1_pred_prob)
ax.plot(fpr1, tpr1, color="coral", linewidth=2.5,
        label=f"Model 1 — Compliance Features         AUC={m1_auc:.4f}")

fpr2a, tpr2a, _ = roc_curve(y2a_clf_test, y2a_clf_pred_prob)
ax.plot(fpr2a, tpr2a, color="seagreen", linewidth=2.5,
        label=f"Model 2A — Center-Level Text           AUC={m2a_clf_auc:.4f}")

fpr2b, tpr2b, _ = roc_curve(center_labels.loc[common_ids],
                              center_clf_simple.loc[common_ids])
ax.plot(fpr2b, tpr2b, color="steelblue", linewidth=2.5, linestyle="--",
        label=f"Model 2B.2 — Individual Reviews        AUC={auc_2b_simple:.4f}")

ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="Random (AUC=0.50)")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves — All 3 Models: Compliance vs Review Text\nModel 1 | Model 2A | Model 2B.2",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10, loc="lower right")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(IMAGE_DIR, "comparison_roc_all_3_models.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f'Saved: {os.path.join(IMAGE_DIR, "comparison_roc_all_3_models.png")}')

**ROC Curve — Inference**

The three curves make the research finding visual. Model 1 (compliance features, orange) barely rises above the random diagonal across the full range of thresholds — it has weak discrimination ability regardless of the operating point chosen. Both text models lift sharply from the origin, confirming strong discrimination ability.

Model 2A (green, AUC=0.8418) achieves the highest lift throughout. Model 2B.2 (blue dashed, AUC=0.8034) sits between the two, well above compliance performance and close to Model 2A. The gap between Model 1 and both text models is consistent across all threshold levels, not concentrated at one point. This means the text advantage holds regardless of how strict or lenient the classification threshold is.

### 3.3 Predicted vs Actual Scatter — All 3 Models

The scatter plots below compare how each model's predictions relate to actual center ratings. A model with strong signal has predictions tracking the red diagonal line. A model with weak signal compresses all predictions into a narrow horizontal band regardless of the actual rating.

In [ ]:
# Display all three scatter plots side by side

models_dir = os.path.join(BASE_DIR, 'images', 'models')
img1 = imread(os.path.join(models_dir, 'm1_xgboost_predicted_vs_actual.png'))
img2 = imread(os.path.join(models_dir, 'm2a_ridge_predicted_vs_actual.png'))
img3 = imread(os.path.join(models_dir, 'm2b_scatter_center.png'))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(img1); axes[0].axis('off')
axes[1].imshow(img2); axes[1].axis('off')
axes[2].imshow(img3); axes[2].axis('off')
plt.suptitle('Predicted vs Actual — All 3 Models',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(IMAGE_DIR, 'comparison_scatter_all_3_models.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {os.path.join(IMAGE_DIR, "comparison_scatter_all_3_models.png")}')

**Predicted vs Actual — Inference**

Model 1 predictions are compressed into a flat band between approximately 4.0 and 4.7 stars regardless of the actual center rating. Centers with actual ratings of 1.0 to 3.0 are predicted in the same range as centers rated 4.5 to 5.0 — the model cannot distinguish genuinely poor centers from excellent ones.

Model 2A predictions track the diagonal most closely. Centers with lower actual ratings receive lower predicted ratings and centers with higher actual ratings receive higher predictions. The model has meaningful discriminative power across the full quality spectrum, consistent with its RMSE=0.4477 and R²=0.3901.

Model 2B.2 sits between the two. Predictions follow the diagonal in the 4.0 to 5.0 range but show more compression than Model 2A — reflecting the label noise from assigning center level ratings to individual reviews. RMSE=0.4933 and R²=0.2597 confirm it is meaningfully better than compliance records (RMSE=0.5555, R²=0.0579) while falling short of center level aggregated text.

The three panels together illustrate the quality gradient: compliance features alone cannot distinguish center quality; individual review text partially can; aggregated center-level text does so most effectively.

### 3.4 City-Level Performance Analysis

The overall AUC and RMSE metrics aggregate across all five cities. This section asks whether the text-over-compliance advantage holds consistently in each city, or whether some cities show larger or smaller gaps. This matters for compliance officers determining where to focus inspection framework reform.

In [ ]:
# Load city labels for test centers — df_model1 not in pkl, read from CSV
df_m1 = pd.read_csv(os.path.join(PROC_DIR, "model1_dataset.csv"))

# Map center IDs to cities
city_map = df_m1.set_index("OPERATION_ID")["CITY"].to_dict()

# Model 1: align city to test centers (340 centers, ordered by test_op_ids)
m1_test_ids   = list(m1["test_op_ids"])
m1_cities     = [city_map.get(op_id, "Unknown") for op_id in m1_test_ids]
df_m1_city = pd.DataFrame({
    "CITY":       m1_cities,
    "true_label": m1["y1_clf_test"],
    "m1_pred":    m1["y1_pred_prob"],
})

# Model 2A: align city to test centers (333 centers, ordered by y2a_test_op_ids)
m2a_test_ids = y2a_test_op_ids
m2a_cities   = [city_map.get(op_id, "Unknown") for op_id in m2a_test_ids]
df_m2a_city = pd.DataFrame({
    "CITY":       m2a_cities,
    "true_label": m2a["y2a_clf_test"],
    "m2a_pred":   m2a["y2a_clf_pred_prob"],
})

# Compute per-city AUC for both models
from sklearn.metrics import roc_auc_score

cities       = sorted(df_m1_city["CITY"].unique())
m1_city_auc  = []
m2a_city_auc = []

for city in cities:
    mask1 = df_m1_city["CITY"] == city
    mask2 = df_m2a_city["CITY"] == city

    y1_t = df_m1_city[mask1]["true_label"];  y1_p = df_m1_city[mask1]["m1_pred"]
    y2_t = df_m2a_city[mask2]["true_label"]; y2_p = df_m2a_city[mask2]["m2a_pred"]

    auc1 = roc_auc_score(y1_t, y1_p) if len(y1_t.unique()) > 1 else float("nan")
    auc2 = roc_auc_score(y2_t, y2_p) if len(y2_t.unique()) > 1 else float("nan")

    m1_city_auc.append(auc1)
    m2a_city_auc.append(auc2)
    print(f"  {city:<15} Model 1={auc1:.4f}  |  Model 2A={auc2:.4f}  |  Gap={auc2-auc1:+.4f}")

# Bar chart
x     = range(len(cities))
width = 0.35
fig, ax = plt.subplots(figsize=(11, 6))
bars1 = ax.bar([i - width/2 for i in x], m1_city_auc,  width, label="Model 1 (Compliance)", color="coral",    edgecolor="white")
bars2 = ax.bar([i + width/2 for i in x], m2a_city_auc, width, label="Model 2A (Text)",      color="seagreen", edgecolor="white")
ax.bar_label(bars1, fmt="%.3f", padding=3, fontsize=9)
ax.bar_label(bars2, fmt="%.3f", padding=3, fontsize=9)
ax.set_xticks(list(x))
ax.set_xticklabels(cities, fontsize=11)
ax.set_ylabel("AUC-ROC", fontsize=11)
ax.set_ylim(0, 1.1)
ax.axhline(y=0.5, color="grey", linestyle="--", linewidth=0.8, label="Random (AUC=0.5)")
ax.set_title("Per-City AUC — Model 1 (Compliance) vs Model 2A (Review Text)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(IMAGE_DIR, "comparison_city_auc.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f'Saved: {os.path.join(IMAGE_DIR, "comparison_city_auc.png")}')

**City-Level Analysis — Inference**

Model 2A (review text) outperforms Model 1 (compliance records) in all five cities — the finding is not driven by any single city or local anomaly.

**Key findings by city:**

- Houston shows the largest text advantage (+0.28). Compliance records achieve AUC=0.550, barely above random — inspection data in Houston carries almost no signal about how families rate daycare quality
- Fort Worth has the highest text model AUC (0.878) and one of the largest compliance gaps (+0.26)
- Austin and Dallas show consistent text advantages of +0.18 and +0.15 respectively
- San Antonio is the closest comparison — Model 1 achieves its best per-city AUC (0.804) while the text advantage is smallest (+0.017), suggesting inspection records are relatively more informative here

**Implications for stakeholders:**

- For compliance officers: Houston should be the highest priority for inspection framework reform — compliance records there provide almost no signal about center quality as experienced by families
- For parents in Houston: checking compliance records is particularly unreliable; reading reviews is even more important than in other cities
- For parents across all cities: the text model advantage holds everywhere — reviews are consistently more reliable than compliance records regardless of city

## 4. What Each Framework Measures

### 4.1 Compliance Framework vs Google Review Text Framework

In [ ]:
# Side-by-side: SHAP (compliance framework) vs TF-IDF word coefficients (Google review framework)

models_dir = os.path.join(BASE_DIR, 'images', 'models')
img_shap = imread(os.path.join(models_dir, 'm1_shap_importance.png'))
img_words = imread(os.path.join(models_dir, 'm2a_word_coefficients.png'))

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
axes[0].imshow(img_shap); axes[0].axis('off')
axes[0].set_title('Compliance Framework\nWhat HHSC Records Measure (SHAP — Model 1)',
                  fontsize=12, fontweight='bold', pad=10)
axes[1].imshow(img_words); axes[1].axis('off')
axes[1].set_title('Google Review Text Framework\nWhat Families Write About (TF-IDF — Model 2A)',
                  fontsize=12, fontweight='bold', pad=10)
plt.suptitle('Two Frameworks — What Each Measures',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(IMAGE_DIR, 'comparison_frameworks_shap_vs_tfidf.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {os.path.join(IMAGE_DIR, "comparison_frameworks_shap_vs_tfidf.png")}')

The core finding is not just that text outperforms compliance numerically — it is that the two frameworks are measuring fundamentally different aspects of a daycare center.

**What HHSC compliance records measure (from Model 1 SHAP analysis):**

The top SHAP features from the compliance model are operational proxies — YEARS_IN_OPERATION (0.3179), TOTAL_INSPECTIONS (0.2790), and ACCEPTS_SUBSIDIES (0.1852). These reflect how long a center has operated and whether it serves subsidized families, not how staff treat children or how families experience daily care. The number of deficiencies in inspections (HIGH_DEFICIENCY_RATE, 0.1270) is the only direct quality measure in the top features.

**What families write in their Google reviews (from Model 2A TF-IDF analysis):**

The strongest positive predictors are "preschool", "montessori", "welcoming", "community", "loving" — words about program identity, environment, and staff warmth. The strongest negative predictors are "rude", "unprofessional", "horrible" — direct judgments of staff behavior. None of these words appear in inspection reports.

**The gap:** Inspectors measure violation counts and operational history. Families measure whether staff are warm, professional, and engaged. These two frameworks are measuring different things — which is why compliance records explain only 6% of rating variance while review text explains 39%.

### 4.2 Word Coefficients — What Drives Ratings

In [ ]:
# Word coefficients for center-level quality (Model 2B.2)
# Shows which words predict HIGH_RATED vs LOW_RATED centers
# using center-aligned split with google_rating >= 4.5 labels
models_dir = os.path.join(BASE_DIR, 'images', 'models')
img_words_2b2 = imread(os.path.join(models_dir, 'm2b_word_coef_center.png'))

fig, ax = plt.subplots(figsize=(12, 9))
ax.imshow(img_words_2b2)
ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(IMAGE_DIR, 'comparison_word_coef_2b2.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {os.path.join(IMAGE_DIR, "comparison_word_coef_2b2.png")}')

**For daycare operators — what drives HIGH ratings:**

Program identity words dominate: "preschool", "montessori", "welcoming", "beautiful", "spanish". Centers perceived as educationally oriented and community focused receive the highest ratings. Staff warmth words — "loving", "wonderful", "amazing" — are the second strongest signal. The two most actionable levers for an operator are program identity (how the center is perceived and described) and staff warmth (how families experience day-to-day interactions with teachers and directors).

**For daycare operators — what drives LOW ratings:**

"rude" and "unprofessional" are the two strongest negative predictors across both Model 2A and Model 2B.2 — independently trained on different data splits with different labels. The cross-model agreement makes this finding robust: staff professionalism is the single most impactful lever an operator can improve, and the signal holds regardless of how the text is modeled.

**For compliance officers — the vocabulary gap:**

The words that predict ratings ("preschool", "rude", "welcoming", "unprofessional") bear no relationship to the items on an inspection checklist (deficiency counts, violation types, capacity). An inspection framework redesigned to capture staff relationship quality and program identity would better track what families actually experience.

Note: several negative predictors in Model 2B.2 ("bca", "cdc", "pinnacle", "merit") appear to be center names or acronyms that the model learned from center-aligned training. These are a modeling artifact — not generalizable behavioral signals — and are noted as a limitation of the center-aligned approach documented in [03-model2b-individual-reviews.ipynb](03-model2b-individual-reviews.ipynb).

## 5. Findings — Research Question Answer

**Research question:** Do Texas HHSC compliance records and families' Google reviews agree on which daycare centers are high quality — and can a parent rely on compliance records alone to make an informed childcare decision?

**The answer is no.**

**For Parents**

Review text is a substantially more reliable signal than compliance records for identifying excellent daycare centers.

Key findings:
- Compliance records correctly identify excellent centers 64% of the time (AUC=0.6441); review text correctly identifies them 84% of the time (AUC=0.8418)
- Review text predicts star ratings with 19% lower error (RMSE: 0.5555 to 0.4477) and explains 39% of rating variance vs 6% for compliance
- The text advantage holds in all five cities without exception
- Houston is the most extreme case — compliance records achieve AUC=0.550 (barely above random) while review text achieves AUC=0.831

What to do: Read even a handful of reviews before choosing a center. When reviews are unavailable, a clean inspection record is a weak positive signal but not a substitute for family experience.

**For Compliance Officers and Policymakers**

The inspection framework measures what inspectors count, not what families experience.

Key findings:
- Compliance records explain only 6% of rating variance; review text explains 39%
- The top SHAP predictors are operational proxies (years in operation, total inspections, subsidy acceptance) not direct quality measures
- Words families use to describe excellent and poor centers (welcoming, preschool, rude, unprofessional) bear no relationship to inspection checklist items
- Per-city analysis shows Houston compliance records are barely above random (AUC=0.550) — the largest misalignment between inspection data and family experience across all five cities

What to do: A review of inspection criteria informed by what families write about would close this gap. Houston should be the highest priority city for reform.

**For Daycare Operators**

Two findings are directly actionable regardless of city.

Key findings:
- "rude" and "unprofessional" are the two strongest negative word predictors across all text models — staff behavior drives poor ratings more than any other factor
- "preschool", "montessori", "welcoming", and "community" are the strongest positive predictors — program identity and environment matter
- These patterns are consistent across both Model 2A and Model 2B.2, independently trained on different approaches

What to do: Invest in staff communication and relationship quality first. Program identity — how the center describes and presents itself — is the second most impactful lever. Compliance improvements alone are insufficient to move ratings.

## 6. Limitations

**Data limitations**

The Google Places API returns at most 5 reviews per center. This API cap limits the benefit of recency weighting (5 data points are too few for temporal weighting to add signal over simple mean) and means that centers with many negative reviews in their full history may appear better than they are in our 5-review sample.

The API returns the 5 most recent reviews. Centers with many older negative reviews may appear better than their full quality history suggests. The 5-review sample may not represent a center's full quality trajectory over time.

The project treats google_rating as ground truth for family experience. Google's aggregation algorithm — which weights by recency, account trustworthiness, and helpfulness votes — is proprietary and cannot be replicated. This introduces an unknown degree of measurement error in the target variable itself.

**Modeling limitations**

The center-aligned Model 2B.2 uses noisy labels — each individual review is labeled with its center's aggregate google_rating. A 1-star review from a 4.8-star center is labeled HIGH_RATED. This label noise limits Model 2B.2's performance compared to Model 2A which uses cleaner center-level aggregated text.

**Analysis scope limitations**

The city-level performance analysis in Section 3.4 compared Model 1 and Model 2A only. Per-city analysis for Model 2B.2 was not conducted as it would require storing city labels for the individual review test centers — this is identified as future work in Section 6. The city finding is therefore based on center-level text modeling only.

## 7. Future Work

**Research and data**

- Expand the review dataset beyond the 5-review API cap per center. Collecting a center's full review history would reduce label noise, improve recency weighting, and give the text models more signal to learn from
- Conduct temporal analysis to understand how center ratings change over time as staff turn over, ownership changes, or centers make deliberate quality improvements

**Modeling**

- Compare XGBoost against a Random Forest baseline for Model 1 to empirically validate the sequential boosting advantage on weak-signal tabular data
- Test TF-IDF models against contextual language models (BERT, sentence transformers) to determine how much additional predictive power comes from word order and semantic context

**Geographic scope**

- Expand beyond Texas's five largest cities to test whether the compliance-vs-text finding generalises to smaller cities and rural centers
- Build a hierarchical model that formally separates within-city and between-city effects to address the city-level confounding identified in the SHAP interaction analysis

**City-level analysis**

- Conduct per-city performance analysis for Model 2B.2 (individual reviews, center-aligned). Section 3.4 compared cities for Model 1 and Model 2A only — extending this to Model 2B.2 would complete the city-level picture

**Productionization**

- Build a parent-facing prediction tool that accepts a center's compliance profile or a set of review texts and returns an instant quality prediction

The findings from this research provide a foundation for a parent-facing childcare discovery product that extends beyond quality prediction to address additional unmet needs in the daycare search process.